In [1]:
%load_ext autoreload
%autoreload 2
%xmode verbose

save_path = 'CTOI Creation/Results'
parent_folder = 'CTOI Creation'

Exception reporting mode: Verbose


In [2]:
new_uploads = [451483379]
update_new_ctois = True
new_host_stars = True
new_transit_models = True

'''
If you want the full thing, put it into a new upload. The full thing also gets host star parameters which takes a lot of time.
If you already have the model stuff, simply don't put it into the new upload and it recalculates parameters from what it has.
The only reason why you should use new upload for the model is if it is new:
if it isn't new, you should have the batman stuff / transit model done / host star parameters and it is already in the ctois.
HOWEVER, if you want to remake the model and not update uploaded_ctois.csv, set the update_ctois to False.
If you don't want host. stars, set it to false too.
And if you don't want transit models (probably because you only want to upload ctois) set it to false.

New: Host star --> attributes --> transit model --> update ctois
Not new: [host star] attributes
'''

from Exoplanet import Exoplanet
from astroquery.mast import Catalogs
import lightkurve
from lightkurve.correctors import DesignMatrix, RegressionCorrector
import numpy
import datetime

data_paths = [] # if you don't qwant to use the files

with open(f'{parent_folder}/queue_data_paths.txt', 'r+') as file:
    data_paths.extend([file_name.strip() for file_name in file.readlines()])
    file.seek(0)

for path in data_paths:
    e = Exoplanet.load_from_json(path)

    NEW = e.id_num in new_uploads

    if NEW:
        print(f'NEW UPLOAD: {e.id_num}!')

        if new_host_stars:
            data = Catalogs.query_object(f'TIC {e.id_num}', catalog = 'TIC')[0]

            host_star = {
                'radius': data['rad'],
                'mass': data['mass'],
                'lum': data['lum'],
                'teff': data['Teff'],
                'logg': data['logg'],
                'ra': data['ra'],
                'dec': data['dec'],
                'logg': data['logg'],
                'feh': data['MH'],
                'J': data['Jmag'],
                'H': data['Hmag'],
                'K': data['Kmag'],
                'Gaia_G': data['GAIAmag'],
                'Tmag': data['Tmag']
            }

            e.add_host_star_attributes(**host_star)

        if new_transit_models:
            search_result = lightkurve.search_targetpixelfile(e.host_star['name'], mission = 'TESS', cadence = 'long')
            pixel_files = search_result.download_all(limit = 10)

            collection = lightkurve.LightCurveCollection(None)
            for pixel_file in pixel_files[:]:
                pixel_file = pixel_file[numpy.isfinite(pixel_file.flux).all(axis = (1, 2))]
                pixel_file = pixel_file[numpy.isfinite(pixel_file.flux_err).all(axis = (1, 2))]

                aperture_mask = pixel_file.pipeline_mask
                uncorrected_lc = pixel_file.to_lightcurve(aperture_mask = aperture_mask)

                design_matrix = DesignMatrix(pixel_file.flux[:, ~aperture_mask]).pca(5).append_constant()
                lc = RegressionCorrector(uncorrected_lc).correct(design_matrix)
                lc = uncorrected_lc
                lc = lc.normalize().flatten(niters = 10).remove_outliers(10.0)
                collection.append(lc)

            lc = collection.stitch()
            lc = lc.flatten()

            e.calculate_all_parameters(lc) # attributes also done in calculate transit model params because it needs them

        else:
            e.calculate_attributes()

    else:
        e.calculate_attributes() # assumes host star done

    e.make_csv_string()

    e.to_json(f'{save_path}/Data/{e.host_star['name']} Exoplanet {e.letter}.json')

    if update_new_ctois and NEW:
        print(f'NEW UPLOAD (and update_ctois = True): {e.id_num}, saving ctoi & mytarget!')
        with open(f'{parent_folder}/uploaded_ctois.csv', 'a') as file:
            file.write(f'{datetime.datetime.now().strftime('%m-%d-%Y')},{e.id_num},https://exofop.ipac.caltech.edu/tess/target.php?id={e.id_num}\n')\

with open(f'{parent_folder}/queue_data_paths.txt', 'r+') as file:
    file.truncate(0)

f'new_uploads = {new_uploads}' # copy and paste

NEW UPLOAD: 451483379!


Sampling: [_x_true, y_pred]


Output()

Sampling: [_x_true, y_pred]


Output()

NEW UPLOAD (and update_ctois = True): 451483379, saving ctoi & mytarget!


'new_uploads = [451483379]'

In [8]:
'''
Creates file upload for new uploads; copy and past from above.
'''

new_uploads = []

import json
import datetime
import os

with open(f'{save_path}/Data/full.json', 'r') as file:
    data = json.loads(file.read())

planet_params_path = ''
folder = f'''{save_path}/File Uploads/{datetime.datetime.now().strftime('%B, %Y')}'''
if not os.path.exists(folder):
    os.makedirs(folder)

for i in range(1, 100):
    planet_params_path = datetime.datetime.now().strftime(f'''{folder}/params_planet_{datetime.datetime.now().strftime('%Y%m%d')}_{i:03d}.txt''')

    if not os.path.exists(planet_params_path):
        break

with open(planet_params_path, 'w') as file:
    string = ''
    for exoplanet in data:
        if new_uploads != [] and exoplanet['id_num'] not in new_uploads:
            continue
        string += exoplanet['csv_string'] + '\n'

    file.write(string)

In [ ]:
'''
Updates 'full.json' with all exoplanets available in the folder (not False Positives). This is then used for the rest of the stuff to make the data.
'''

import os
import json

data = []
for (dirpath, dirnames, filenames) in os.walk(f'{save_path}/Data'):
    for filepath in filenames:
        if filepath == '.DS_Store' or filepath.endswith('.csv'):
            continue

        with open(dirpath + '/' + filepath, 'r') as file:
            exoplanet = json.loads(file.read())

        if type(exoplanet) == list:
            continue

        data.append(exoplanet)

    break

data = sorted(data, key = lambda x: x['name'])
# with open(f'{parent_folder}/ctoi_targets.txt', 'a') as file:
#     for e in data:
#         file.write(f'{e['id_num']}\n')

with open(f'{save_path}/Data/full.json', 'w') as file:
    string = json.dumps(data, indent = 4)
    string = string.replace('NaN', 'null')
    string = string.replace('nan', '')

    file.write(string)

In [5]:
import json
import pandas
import math

with open(f'{save_path}/Data/full.json', 'r') as file:
    data = json.loads(file.read())

data = pandas.DataFrame([exoplanet['parameters'] for exoplanet in data])
data['target'] = data['target'].apply(lambda x: x.replace('TIC', 'TIC '))
data = data.set_index('target')
data.replace('', math.nan, inplace = True)
data = data.drop(['tag', 'flag'], axis = 1)
data['disp'] = data['disp'].mask(data['mass'] >= 13 * 317.9, 'FP')
data = data.sort_values(['disp', 'target'
                         ], ascending = [False, True])

data = data[['disp', 'period', 'period_unc', 'epoch', 'epoch_unc', 'r_planet', 'radius', 'mass', 'mass_unc_lower', 'mass_unc_higher', 'temp', 'depth', 'depth_unc', 'duration', 'duration_unc', 'sma', 'imp', 'inc', 'arg_peri', 'ecc', 'u1', 'u2']]
data['epoch'] -= 2457000
columns = data.select_dtypes(include = 'number').columns.to_list()

for value in ['period', 'depth', 'duration', 'epoch']:
    columns.remove(value)

data[columns] = data[columns].map(
    lambda x: x if x == 0 or math.isnan(x) else round(x, (3 - 1) - int(math.floor(math.log10(abs(x)))))
)

data[['period', 'depth', 'duration', 'epoch']] = data[['period', 'depth', 'duration', 'epoch']].map(
    lambda x: x if x == 0 or math.isnan(x) else round(x, (6 - 1) - int(math.floor(math.log10(abs(x)))))
)

data.replace(math.nan, 'null', inplace = True)

data.index.names = ['Target (CTOI ID)']
data.index = data.index.str.replace('TIC ', '', regex = False)
data['Period (days)'] = data['period'].astype(str) + ' ± ' + data['period_unc'].astype(str)
data['Epoch (BJD_TDB - 2457000)'] = data['epoch'].astype(str) + ' ± ' + data['epoch_unc'].astype(str)
data['Transit Depth (PPM)'] = data['depth'].astype(str) + ' ± ' + data['depth_unc'].astype(str)
data['Transit Duration (hours)'] = data['duration'].astype(str) + ' ± ' + data['duration_unc'].astype(str)

data['Disposition'] = data['disp']
data['Planet Radius / Stellar Radius'] = data['r_planet']
data['Radius (R⊕)'] = data['radius']
data['Mass (M⊕)'] = data['mass']
data['Mass Unc (M⊕)'] = '+' + data['mass_unc_higher'].astype(str) + ', -' + data['mass_unc_lower'].astype(str)
data['Equilibrium Temperature (K)'] = data['temp']
data['Semi-major Axis (AU)'] = data['sma']

data['Impact Parameter'] = data['imp']
data['Inclination (deg)'] = data['inc']
data['Eccentricity'] = data['ecc']
data['Argument of Periastron'] = data['arg_peri']
data['Limb Darkening Coefficient 1'] = data['u1']
data['Limb Darkening Coefficient 2'] = data['u2']

data = data[['Period (days)', 'Epoch (BJD_TDB - 2457000)', 'Transit Depth (PPM)', 'Transit Duration (hours)',
             'Disposition', 'Planet Radius / Stellar Radius', 'Radius (R⊕)', 'Mass (M⊕)', 'Mass Unc (M⊕)', 'Equilibrium Temperature (K)', 'Semi-major Axis (AU)',
             'Impact Parameter', 'Inclination (deg)', 'Eccentricity', 'Argument of Periastron', 'Limb Darkening Coefficient 1', 'Limb Darkening Coefficient 2']]

data.to_csv(f'{save_path}/ctois.csv')

In [6]:
import json
import pandas
import math

with open(f'{save_path}/Data/full.json', 'r') as file:
    data = json.loads(file.read())

data = pandas.DataFrame([exoplanet['host_star'] for exoplanet in data])
data = data.set_index('name')
data.replace('', math.nan, inplace = True)
data = data.sort_index()

data = data[['dec', 'logg', 'lum', 'mass', 'ra', 'radius', 'teff']]
columns = data.select_dtypes(include = 'number').columns.to_list()

for value in ['ra', 'dec']:
    columns.remove(value)

data[columns] = data[columns].map(
    lambda x: x if x == 0 or math.isnan(x) else round(x, (3 - 1) - int(math.floor(math.log10(abs(x)))))
)

data[['ra', 'dec']] = data[['ra', 'dec']].map(
    lambda x: x if x == 0 or math.isnan(x) else round(x, (6 - 1) - int(math.floor(math.log10(abs(x)))))
)

data.replace(math.nan, 'null', inplace = True)

data.index.names = ['Target (TIC ID)']
data.index = data.index.str.replace('TIC ', '', regex = False)

data['Declination (deg)'] = data['dec']
data['Right Ascension (deg)'] = data['ra']
data['Radius (R⊙)'] = data['radius']
data['Mass (M⊙)'] = data['mass']
data['Surface Gravity (log(g))'] = data['logg']
data['Effective Temperature (K)'] = data['teff']

data = data[['Declination (deg)', 'Right Ascension (deg)', 'Radius (R⊙)', 'Mass (M⊙)', 'Surface Gravity (log(g))', 'Effective Temperature (K)']]

data.to_csv(f'{save_path}/stars.csv')

In [7]:
'''
This chekcs what I have uploaded.
'''

import pandas
import json

with open(f'{save_path}/Data/full.json', 'r') as file:
    data = json.loads(file.read())

ticids = [exoplanet['id_num'] for exoplanet in data]

data = pandas.DataFrame()
for ticid in ticids:
    url = f'https://exofop.ipac.caltech.edu/tess/download_planet.php?id={ticid}'

    data = pandas.concat([data, pandas.read_csv(url, delimiter = '|')])

data = data.sort_values('Name')
data = data[data['Table'] == 'CTOI'].drop_duplicates('Name')
# data = data[['Name', 'TOI', 'Disposition', 'Date', 'User', 'Tag']]

data

,Table,ID,Name,TOI,Disposition,Epoch (BJD),Epoch (BJD) Error,Period (days),Period (days) Error,Depth (mmag),...,Time of Periastron (BJD),Time of Periastron (BJD) Error,Velocity Semi-amplitude (m/s),Velocity Semi-amplitude (m/s) Error,SNR,Date,User,Group,Tag,Notes
